# Lab 0 - Setup

**Octank Gift Concierge** - you'll take a small gift-shopping model from raw data to a deployed
custom model: generate data, profile it, clean it, label it, review edge cases, fill coverage
gaps, prepare SFT and RLVR datasets, train **SFT, then SFT -> RLVR** with **serverless model
customization**, evaluate with **LLM-as-a-Judge**, and serve it behind a Streamlit demo.

| Lab | Notebook |
|---|---|
| 0 | Setup (this notebook) - and start your endpoint early |
| 1 | `01_generate_profile.ipynb` - generate ~1,000 raw requests, profile them |
| 2 | `02_curate.ipynb` - dedup, toxicity, auto-label, human review, gap-fill |
| 3 | `03_prepare_train.ipynb` - build SFT + RLVR datasets, review the reward function, train SFT then SFT -> RLVR |
| 4 | `04_evaluate_deploy.ipynb` - deploy adapters, live checks, Streamlit demo |
| 5 | `05_compare.ipynb` - compare your models with the instructor's, by problem type |

Everything runs in **your current region** and writes to the SageMaker default bucket under
`s3://sagemaker-<region>-<account>/gift-concierge/attendee/`.

In [ ]:
# Pinned SDKs (latest as of 2026-10-28). Restart the kernel after the first install.
%pip install -q -r ../requirements.txt

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath("../src"))
%load_ext autoreload
%autoreload 2
import pandas as pd
pd.set_option("display.max_colwidth", 120)
from octank import config, state
S = config.get_settings("attendee")
ST = state.load(S)
print(f"region={S.region}  bucket={S.bucket}  prefix={S.prefix}  run={S.run_name}")
print(f"role={S.role_arn}")

## Check model access
Every model that touches training data is open-weight on Bedrock (no deployment needed). Claude Haiku 4.5 is used only as the evaluation judge.

In [ ]:
from octank import llm
checks = {
    "generator  (gpt-oss-120b)": config.GENERATOR_MODEL,
    "labeler    (gpt-oss-20b)": config.LABELER_MODEL,
    "safety     (gpt-oss-safeguard-20b)": config.SAFETY_MODEL,
    "reward taste (gpt-oss-20b)": config.TASTE_MODEL,
    "eval judge (Claude Haiku 4.5)": config.judge_profile_id(S.region),
}
for name, mid in checks.items():
    try:
        r = llm.converse(mid, "Reply with the single word: ready", max_tokens=200, temperature=0,
                         region=S.region)
        print(f"OK   {name:38s} -> {r.text[:30]!r}")
    except Exception as e:
        print(f"FAIL {name:38s} -> {type(e).__name__}: {str(e)[:120]}")

## Start your endpoint now (it takes a while)

GPU capacity can take time, so we create the `ml.g7.2xlarge` endpoint with the **base model**
now; Lab 4 attaches the instructor's fine-tuned adapters (and yours, if they've finished). This
uses **sagemaker-core** resources
(`Model`, `EndpointConfig`, `Endpoint`, `InferenceComponent`). It is billed hourly until you run
the cleanup cell at the end of Lab 4.

In [ ]:
from octank import serving
ENDPOINT = ST.get("endpoint") or "octank-concierge"
state.update(S, endpoint=ENDPOINT, base_ic=f"{ENDPOINT}-base-ic")

print(f"Creating {ENDPOINT} - this cell runs until the endpoint and base model are InService.")
try:
    serving.create_endpoint(ENDPOINT, S.role_arn, S.region)
    out = serving.add_model(ENDPOINT, S.role_arn, S.region)
    state.update(S, endpoint_status="InService", **out)
    print("endpoint ready:", out)
except Exception as e:
    state.update(S, endpoint_status=f"FAILED: {e}")
    print("endpoint failed:", e)
    raise